# Prompt Engineering Showdown

This notebook demonstrates three prompt engineering techniques: zero-shot, few-shot and chain-of-thought prompting. The techniques are tested on sentiment classification, entity extraction and text summarisation using Ollama with the qwen3:0.6b model. The notebook also demonstrates a prompt failure mode and fixes inconsistent output using Pydantic structured output. The results are compared to understand how different prompting strategies affect model responses.

## 1. Setup and Ollama Connection

In [15]:
import requests

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen3:0.6b"

def ask_ollama(prompt, output_format=None):
    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "stream": False
    }

    if output_format:
        payload["format"] = output_format

    response = requests.post(OLLAMA_URL, json=payload)
    response.raise_for_status()

    return response.json()["message"]["content"]

## 2. Task 1 - Sentiment Classification
### 2.1 Zero-shot

In [16]:
import requests

prompt = """
Classify the sentiment of the following text as Positive, Negative or Neutral.

Text: "I absolutely loved this restaurant. The food was delicious and the service was excellent."

Return only the sentiment label.
"""

result = ask_ollama(prompt)

print("Zero-shot Output:")
print(result)
sentiment_zero_shot = result

Zero-shot Output:
Positive


### 2.2 Few-shot

In [17]:
prompt = """
Classify the sentiment as Positive, Negative or Neutral.

Examples:

Text: "This product is amazing and I am very happy with it."
Sentiment: Positive

Text: "The service was terrible and I am very disappointed."
Sentiment: Negative

Text: "The package arrived yesterday."
Sentiment: Neutral

Now classify this text:

Text: "I absolutely loved this restaurant. The food was delicious and the service was excellent."

Return only the sentiment label.
"""

result = ask_ollama(prompt)

print("Few-shot Output:")
print(result)
sentiment_few_shot = result

Few-shot Output:
Positive


### 2.3 Chain-of-Thought

In [18]:
prompt = """
Classify the sentiment of the text as Positive, Negative, or Neutral.

First, briefly explain the main clues that indicate the sentiment.
Then provide the final sentiment label.

Text: "I absolutely loved this restaurant. The food was delicious and the service was excellent."

Use this format:

Reason: brief explanation
Sentiment: Positive, Negative, or Neutral
"""

result = ask_ollama(prompt)

print("Chain-of-Thought Output:")
print(result)
sentiment_cot = result

Chain-of-Thought Output:
Reason: The text explicitly states positive sentiments ("loved," "delicious," "excellent") and conveys satisfaction with the restaurant's features.  
Sentiment: Positive.


## 3. Task 2 - Entity Extraction
### 3.1 Zero-shot

In [19]:
prompt = """
Extract the important named entities from the following text.

Text:
"Microsoft CEO Satya Nadella visited Bengaluru, India, in March 2026."

Identify the person, organization, location and date.

Return the entities with their types.
"""

result = ask_ollama(prompt)

print("Zero-shot Entity Extraction:")
print(result)
entity_zero_shot = result

Zero-shot Entity Extraction:
- Person: Satya Nadella (Person)  
- Organization: Microsoft (Organization)  
- Location: Bengaluru, India (Location)  
- Date: March 2026 (Date)


### 3.2 Few-shot

In [20]:
prompt = """
Extract named entities from the text and identify their types.

Use these examples:

Example 1:
Text: "Elon Musk founded SpaceX in the United States."
Entities:
Person: Elon Musk
Organization: SpaceX
Location: United States

Example 2:
Text: "Tim Cook announced a new Apple product in California."
Entities:
Person: Tim Cook
Organization: Apple
Location: California

Example 3:
Text: "Sundar Pichai spoke at Google headquarters in California."
Entities:
Person: Sundar Pichai
Organization: Google
Location: California

Now extract entities from:

Text: "Microsoft CEO Satya Nadella visited Bengaluru, India, in March 2026."

Return the entities with their types.
"""

result = ask_ollama(prompt)

print("Few-shot Entity Extraction:")
print(result)
entity_few_shot = result

Few-shot Entity Extraction:
Entities:  
- Person: Satya Nadella  
- Organization: Microsoft  
- Location: India  
- Location: Bengaluru


### 3.3 Chain-of-Thought

In [21]:
prompt = """
Extract named entities from the text.

First, briefly identify the important names, organizations, places and dates in the text.
Then classify each entity by type.

Text:
"Microsoft CEO Satya Nadella visited Bengaluru, India, in March 2026."

Use this format:

Reason: brief explanation
Entities:
Person: ...
Organization: ...
Location: ...
Date: ...
"""

result = ask_ollama(prompt)

print("Chain-of-Thought Entity Extraction:")
print(result)
entity_cot = result

Chain-of-Thought Entity Extraction:
Reason: The text provides information about a Microsoft CEO visiting a specific location in India.  
Entities:  
Person: Satya Nadella  
Organization: Microsoft  
Location: Bengaluru, India  
Date: March 2026


## 4. Task 3 - Text Summarisation
### 4.1 Zero-shot

In [22]:
prompt = """
Summarize the following text in 2-3 sentences.

Text:
"Artificial intelligence is transforming many industries, including healthcare,
finance, education and transportation. AI systems can analyze large amounts of
data, automate repetitive tasks and help people make better decisions. However,
AI also creates challenges such as privacy concerns, bias, job displacement and
the need for responsible development."
"""

result = ask_ollama(prompt)

print("Zero-shot Summary:")
print(result)
summary_zero_shot = result

Zero-shot Summary:
Artificial intelligence is transforming industries such as healthcare, finance, and education, enabling automation and data-driven decision-making. However, it also raises concerns about privacy, bias, job displacement, and the need for responsible development.


### 4.2 Few-shot

In [23]:
prompt = """
Summarize the text in 2-3 sentences. Focus only on the main ideas.

Example 1:
Text: "Electric vehicles use electric motors instead of traditional gasoline engines.
They can reduce fuel costs and tailpipe emissions, although charging infrastructure
and battery costs remain challenges."

Summary: Electric vehicles use electric motors and can reduce fuel costs and emissions.
However, charging infrastructure and battery costs remain challenges.

Example 2:
Text: "Remote work allows employees to work from locations outside the traditional
office. It can improve flexibility and reduce commuting, but communication and
team collaboration can sometimes be difficult."

Summary: Remote work provides greater flexibility and eliminates commuting for many
employees. However, communication and collaboration can be challenging.

Now summarize this text:

Text: "Artificial intelligence is transforming many industries, including healthcare,
finance, education and transportation. AI systems can analyze large amounts of
data, automate repetitive tasks and help people make better decisions. However,
AI also creates challenges such as privacy concerns, bias, job displacement and
the need for responsible development."

Summary:
"""

result = ask_ollama(prompt)

print("Few-shot Summary:")
print(result)
summary_few_shot = result

Few-shot Summary:
Artificial intelligence is transforming industries such as healthcare, finance, education, and transportation by automating tasks and improving decision-making, yet it faces challenges like privacy concerns, bias, job displacement, and the need for responsible development.


### 4.3 Chain-of-Thought

In [24]:
prompt = """
Summarize the following text in 2-3 sentences.

First, briefly identify the main ideas and important challenges.
Then write a concise summary that includes only the most important information.

Text:
"Artificial intelligence is transforming many industries, including healthcare,
finance, education and transportation. AI systems can analyze large amounts of
data, automate repetitive tasks and help people make better decisions. However,
AI also creates challenges such as privacy concerns, bias, job displacement and
the need for responsible development."

Format your answer as:

Main ideas: <brief points>
Summary: <2-3 sentence summary>
"""

result = ask_ollama(prompt)

print("Chain-of-Thought Summary:")
print(result)
summary_cot = result

Chain-of-Thought Summary:
Main ideas: AI's transformative impact across healthcare, finance, education, and transportation, alongside challenges like privacy, bias, job displacement, and ethical development.  
Summary: Artificial intelligence is revolutionizing industries but raises concerns over privacy, bias, job displacement, and the need for responsible innovation.


## 5. Prompt Failure Mode - Inconsistent Output Format

In [25]:
prompt = """
Extract information from this text.

Text:
"Microsoft CEO Satya Nadella visited Bengaluru, India, in March 2026."

Give me the important information.
"""

result = ask_ollama(prompt)

print("Failure Mode Output:")
print(result)

Failure Mode Output:
The important information extracted from the text is: Microsoft CEO Satya Nadella visited Bengaluru, India, in March 2026.


### Diagnosis

The prompt does not specify a fixed output format or required fields. Because the instructions are open-ended, the model may return the information in different formats or include unnecessary details. This makes the output inconsistent and difficult to process programmatically.

To fix this issue, a Pydantic model is used to define the required fields and Ollama's structured-output `format` parameter is used to request JSON that follows the defined schema.


## 6. Structured Output with Pydantic

In [26]:
from pydantic import BaseModel
import requests
import json

# Define the required output structure
class EntityOutput(BaseModel):
    person: str
    organization: str
    location: str
    date: str

# Create the JSON schema for Ollama
schema = EntityOutput.model_json_schema()

prompt = """
Extract the entities from this text.

Text:
"Microsoft CEO Satya Nadella visited Bengaluru, India, in March 2026."

Return the information according to the provided JSON schema.
"""

result = ask_ollama(prompt, schema)

print("Structured JSON Output:")
print(result)

Structured JSON Output:
{

"person": "Satya Nadella",
"organization": "Microsoft",
"location": "Bengaluru, India",
"date": "March 2026"
}


In [27]:
parsed = EntityOutput.model_validate_json(result)

print("Parsed Pydantic Object:")
print(parsed)

Parsed Pydantic Object:
person='Satya Nadella' organization='Microsoft' location='Bengaluru, India' date='March 2026'


## 7. Prompting Technique Comparison

In [28]:
from IPython.display import display
import pandas as pd

comparison = pd.DataFrame({
    "Task": [
        "Sentiment Classification",
        "Entity Extraction",
        "Text Summarisation"
    ],
    "Zero-shot": [
        sentiment_zero_shot,
        entity_zero_shot,
        summary_zero_shot
    ],
    "Few-shot": [
        sentiment_few_shot,
        entity_few_shot,
        summary_few_shot
    ],
    "Chain-of-thought": [
        sentiment_cot,
        entity_cot,
        summary_cot
    ]
})

display(comparison)

,Task,Zero-shot,Few-shot,Chain-of-thought
0,Sentiment Classification,Positive,Positive,Reason: The text explicitly states positive se...
1,Entity Extraction,- Person: Satya Nadella (Person) \n- Organiza...,Entities: \n- Person: Satya Nadella \n- Orga...,Reason: The text provides information about a ...
2,Text Summarisation,Artificial intelligence is transforming indust...,Artificial intelligence is transforming indust...,Main ideas: AI's transformative impact across ...


### Observation

The results do not always improve progressively. In the entity extraction task, few-shot prompting actually missed some entities. This shows that prompt engineering can improve consistency, but small models can still produce unexpected results.

## 8. Conclusion

Zero-shot prompting is simple but gives the model less guidance. Few-shot prompting provides examples that help the model understand the expected task and output pattern. Chain-of-thought prompting can encourage the model to organize its reasoning before producing an answer. The failure-mode experiment shows that ambiguous instructions can lead to inconsistent outputs. Using Pydantic with Ollama's structured-output format makes the response predictable, validated and easier to process programmatically.